# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwahab-git/week-01-Assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Signal 1: Content staleness
# Flag-linked signal: refresh/staleness

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 89, 179, 364, np.inf],
    labels=["<90 days", "90-179 days", "180-364 days", "365+ days"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_clicks=("clicks_90d", "median")
      )
      .reset_index()
)

print("Signal: days_since_last_update")
print(staleness_check.to_string(index=False))


Signal: days_since_last_update
staleness_bucket     n  median_impressions  median_clicks
        <90 days 20655               472.0            1.0
     90-179 days  9171              1692.0            2.0
    180-364 days   169                16.0            0.0
       365+ days     5                 2.0            0.0


In [19]:
# Signal 2: Existing visibility / volume
# Flag-linked signal: volume / quick-win

df["volume_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

volume_check = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_clicks=("clicks_90d", "median"),
          median_sessions=("sessions_90d", "median"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

print("Signal: impressions_90d")
print(volume_check.to_string(index=False))

Signal: impressions_90d
volume_bucket    n  median_clicks  median_sessions  median_ctr
          Low 7503            0.0              2.0        0.00
       Medium 7499            0.0              4.0        0.00
         High 7498            2.0             11.0        0.13
    Very High 7500           22.0             54.0        0.21


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Build the ranked action queue

# Transparent score:
# More existing impressions + more existing clicks = higher priority
df["score"] = df["impressions_90d"] * (1 + df["clicks_90d"])

# One reason code
df["reason_code"] = np.where(
    df["score"] > 0,
    "existing_visibility",
    "no_existing_visibility"
)

# Action label
df["action"] = np.where(
    df["score"] > 0,
    "REVIEW",
    "NO_ACTION"
)

# Rank highest score first
queue = (
    df[
        [
            "content_id",
            "client_id",
            "score",
            "reason_code",
            "action",
            "impressions_90d",
            "clicks_90d"
        ]
    ]
    .sort_values(
        ["score", "impressions_90d", "clicks_90d"],
        ascending=False
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

# Put rank first
queue = queue[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "impressions_90d",
        "clicks_90d"
    ]
]

# Write the required CSV
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("CSV written:", output_path)
print("Rows:", len(queue))

print("\nTop 10:")
display(queue.head(10))


CSV written: work/outputs/baseline_action_score.csv
Rows: 30000

Top 10:


,rank,content_id,client_id,score,reason_code,action,impressions_90d,clicks_90d
0,1,content_07e0b9af8b1a,client_b4944c6ff0,897716064,existing_visibility,REVIEW,214816,4178
1,2,content_4c36c775b818,client_4e07408562,875264670,existing_visibility,REVIEW,463103,1889
2,3,content_9532f197bbc8,client_4e07408562,831726480,existing_visibility,REVIEW,309192,2689
3,4,content_8e7ba84a972b,client_7f2253d7e2,765482604,existing_visibility,REVIEW,288426,2653
4,5,content_89e84d699e9e,client_349c41201b,677331186,existing_visibility,REVIEW,275226,2460
5,6,content_aaef01a50def,client_19581e27de,657245539,existing_visibility,REVIEW,517109,1270
6,7,content_2c2606c5d176,client_19581e27de,644425145,existing_visibility,REVIEW,347399,1854
7,8,content_44e481c8f55b,client_19581e27de,635706902,existing_visibility,REVIEW,312694,2032
8,9,content_2dba2b1f9536,client_6208ef0f77,403968374,existing_visibility,REVIEW,443434,910
9,10,content_8c19996aa890,client_4e07408562,400272072,existing_visibility,REVIEW,509252,785


### Baseline rule

**Rule:** Prioritize content that already has measurable visibility and engagement.

The score is:

`score = impressions_90d × (1 + clicks_90d)`

This is intentionally transparent and uses no fitted weights. Higher existing impressions and clicks produce a higher review priority.

**Reason code:** `existing_visibility`

**Action:** `REVIEW` when the score is greater than zero; otherwise `NO_ACTION`.

The first signal check showed that staleness (`days_since_last_update`) was OPPOSITE to the expected refresh signal in this dataset, so staleness is not used as a positive scoring feature. The volume check using `impressions_90d` was CONFIRMED and therefore forms the basis of this baseline.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

| Rank | Action | Reason code | Confidence | Why it's there | What would make it wrong |
|---:|---|---|---|---|---|
| 1 | REVIEW | existing_visibility | High | Very high impressions and clicks produce the highest score. | The traffic could be irrelevant, low-value, or already fully optimized. |
| 2 | REVIEW | existing_visibility | High | Strong impressions and clicks give this item substantial existing visibility. | High traffic may leave little practical optimization opportunity. |
| 3 | REVIEW | existing_visibility | High | Strong impressions combined with high clicks produce a very high score. | The clicks may come from queries that are not relevant or actionable. |
| 4 | REVIEW | existing_visibility | High | High impressions and clicks indicate established visibility and engagement. | The page may already be performing well enough that no action is needed. |
| 5 | REVIEW | existing_visibility | High | Strong existing impressions and clicks place this item near the top of the queue. | Its traffic may be valuable but not represent an optimization opportunity. |
| 6 | REVIEW | existing_visibility | High | The highest impression count among the top group combined with substantial clicks gives a high score. | High impressions with comparatively fewer clicks may indicate that visibility does not translate into useful engagement. |
| 7 | REVIEW | existing_visibility | High | Strong impressions and clicks indicate meaningful existing visibility. | The traffic may not be relevant to the intended action. |
| 8 | REVIEW | existing_visibility | High | High impressions and clicks produce a strong score. | The content may already be adequately optimized. |
| 9 | REVIEW | existing_visibility | High | Very high impressions and meaningful clicks make this a strong queue candidate. | High visibility may not mean there is a useful change to make. |
| 10 | REVIEW | existing_visibility | High | Very high impressions combined with 785 clicks produce a high score. | The relatively lower click count could indicate weak engagement despite visibility. |
| 11 | REVIEW | existing_visibility | High | Strong impressions and clicks give this item substantial existing visibility. | The observed traffic may not be actionable or relevant. |
| 12 | REVIEW | existing_visibility | High | Very high impressions with meaningful clicks produce a high score. | The page may have visibility without a worthwhile improvement opportunity. |
| 13 | REVIEW | existing_visibility | High | Strong impressions and clicks indicate established traffic. | The traffic may already be performing adequately. |
| 14 | REVIEW | existing_visibility | High | High impressions combined with meaningful clicks place it in the upper part of the queue. | The traffic may not correspond to an optimization opportunity. |
| 15 | REVIEW | existing_visibility | Medium | Very high impressions compensate for relatively fewer clicks and still produce a high score. | The low click count relative to impressions may make this a weak practical pick. |
| 16 | REVIEW | existing_visibility | High | Strong clicks together with substantial impressions produce a high score. | The traffic may already be performing well and require no action. |
| 17 | REVIEW | existing_visibility | High | Strong impressions and clicks indicate meaningful existing visibility. | The traffic may be irrelevant or not actionable. |
| 18 | REVIEW | existing_visibility | High | High impressions and meaningful clicks produce a strong score. | The content may already be optimized sufficiently. |
| 19 | REVIEW | existing_visibility | High | Strong clicks relative to its impressions give this item a high score. | The observed traffic may not represent a useful improvement opportunity. |
| 20 | REVIEW | existing_visibility | High | Strong impressions and clicks place this item in the top 20. | The page may already perform well enough that intervention would not help. |

**Review conclusion:** The rule consistently selects content with existing visibility and engagement. However, the score only identifies pages worth reviewing; it does not prove that an optimization action is needed. Manual review is required before acting.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Weak picks + leakage check

# Identify a potential weak pick:
# high impressions but relatively few clicks.
weak_pick = queue.iloc[14]

print("Potential weak pick:")
print(weak_pick.to_string())

print("\nLeakage check:")

# Features used by the baseline
used_features = [
    "impressions_90d",
    "clicks_90d"
]

# Confirm prohibited trend/label-derived columns were not used
prohibited_features = [
    "trend_pct",
    "trend_direction"
]

print("Features used:", used_features)
print("Prohibited trend features used: NO")

assert not any(
    feature in used_features for feature in prohibited_features
)

print("Leakage check passed.")

Potential weak pick:
rank                                 15
content_id         content_2cb567c3c89b
client_id             client_6208ef0f77
score                         242890776
reason_code         existing_visibility
action                           REVIEW
impressions_90d                  497727
clicks_90d                          487

Leakage check:
Features used: ['impressions_90d', 'clicks_90d']
Prohibited trend features used: NO
Leakage check passed.


### Weak-pick review

Rank 15 is a useful weak pick to inspect. It has very high impressions (497,727) but only 487 clicks. The rule ranks it highly because the score combines impressions and clicks, but high visibility alone does not prove that the content needs optimization. This shows why the ranked queue is a review queue rather than an automatic action list.

### Leakage check

The baseline uses only `impressions_90d` and `clicks_90d`. It does not use `trend_pct` or `trend_direction`, which are prohibited because they are label-derived. No future-window inputs are used.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.